**APEM ATTACK - FIXED**

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 1 — MOUNT DRIVE
# ═══════════════════════════════════════════════════════════════════════════
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 2 — CONFIGURATION
# ─ Edit SEEDS_IID and SEEDS_NONIID to list every seed folder you have.
# ─ Edit BASE_PATH and PT_ROOT if your Drive layout differs.
# ─ APEM_SIGMA_IID / APEM_SIGMA_NONIID: paste the actual per-client final sigma
#   values from your original training logs for each seed.
#   If you do not have them, leave as NOISE_MULTIPLIER (1.6) — conservative fallback.
# ═══════════════════════════════════════════════════════════════════════════

BASE_PATH = '/content/drive/MyDrive/Thesis Dataset'
PT_ROOT   = '/content/drive/MyDrive/Thesis Dataset/results/SELECTED_SEEDS_PT'

# ── Seeds — add all seed folder names exactly as they appear on Drive ─────────
SEEDS_IID    = [27,1165,8965,688,901,155,333,893,420,67]        # e.g. [8695, 27, 42]
SEEDS_NONIID = [27,1165,8965,688,901,155,333,893,420,67]       # e.g. [89655, 31, 77]

# ── Hyperparameters — must be IDENTICAL to original training ──────────────────
NUM_CLIENTS       = 10
LR                = 0.0001
LOCAL_EPOCHS      = 1
CLIP_NORM         = 0.5
NOISE_MULTIPLIER  = 1.6     # Static DP-FL sigma; also fallback for APEM if unknown
ATTACK_BATCH_SIZE = 32
ATTACK_N_TRIALS   = 5

# ── APEM per-client final sigma values ───────────────────────────────────────
# These should come from your original training logs (the per-client noise
# that APEM settled on after 20 rounds). If unknown, leave as NOISE_MULTIPLIER.
# Format: { seed: [sigma_C1, sigma_C2, ..., sigma_C10] }
APEM_SIGMA_IID = {
    27: [1.7400,1.7400,1.7400,1.7400,1.7400,1.7400,1.7400,1.7400,1.7400,1.7400],
    1165: [1.7775,1.7775,1.7775,1.7775,1.7775,1.775,1.775,1.775,1.775,1.775],
    8965: [1.7650,1.7650, 1.7650, 1.7650, 1.7650, 1.7650, 1.7650, 1.7650, 1.7650, 1.7650],
    688: [1.7525,1.7525,1.7525,1.7525,1.7525,1.7525,1.7525,1.7525,1.7525,1.7525],
    901: [1.7462,1.7462,1.7462,1.7462,1.7462,1.7462,1.7462,1.7462,1.7462,1.7462],
    155: [1.7688,1.7688,1.7688,1.7688,1.7688,1.7688,1.7688,1.7688,1.7688,1.7688],
    333: [1.7587,1.7587,1.7587,1.7587,1.7587,1.7587,1.7587,1.7587,1.7587,1.7587],
    893: [1.7487,1.7487,1.7487,1.7487,1.7487,1.7487,1.7487,1.7487,1.7487,1.7487],
    420: [1.7600,1.7600,1.7600,1.7600,1.7600,1.7600,1.7600,1.7600,1.7600,1.7600],
    67: [1.7560,1.7560,1.7560,1.7560,1.7560,1.7560,1.7560,1.7560,1.7560,1.7560]


    # replace with real values
    # 27: [1.6, 1.6, 1.6, 1.6, 1.6, 1.6, 1.6, 1.6, 1.6, 1.6],
}
APEM_SIGMA_NONIID = {
    8965: [1.5900,1.6112,1.5713,2.0000,1.5538,1.7337,2.0000,1.7837,1.5775,1.7738],
    27: [1.6013,1.6300,1.5863,2.0000,1.5688,1.7187,1.9637,1.7537,1.5963,1.7462],
    1165: [1.5938,1.5925,1.5600,2.0000,1.5538,1.7375,2.0000,1.7850,1.5850,1.7750],
    901: [1.5975,1.6187,1.5788,2.0000,1.5613,1.7162,1.9862,1.7612,1.5888,1.7537],
    155: [1.5938,1.6075,1.5675,2.0000,1.5538,1.7325,2.0000,1.7800,1.5775,1.7700],
    333: [1.5975,1.6187,1.5825,2.0000,1.5688,1.7237,2.0000,1.7687,1.5925,1.7563],
    893: [1.5975,1.6112,1.5750,2.0000,1.5613,1.7237,1.9987,1.7662,1.5888,1.7587],
    420: [1.5975, 1.6150, 1.5750, 2.000, 1.5575, 1.7187, 2.000, 1.7638, 1.5850, 1.7587],
    67: [1.6050, 1.6150,1.5750, 2.000, 1.5613, 1.7187, 1.9862, 1.7612, 1.5850,1.7535],
    688: [1.5975, 1.6150,1.5825, 2.000, 1.5613, 1.7225, 2.000, 1.7700, 1.5888, 1.7625]

    # replace with real values
    # 31:  [1.6, 1.6, 1.6, 1.6, 1.6, 1.6, 1.6, 1.6, 1.6, 1.6],
}

print('Configuration done.')
print(f'  IID seeds    : {SEEDS_IID}')
print(f'  Non-IID seeds: {SEEDS_NONIID}')
print(f'  PT root      : {PT_ROOT}')

Configuration done.
  IID seeds    : [27, 1165, 8965, 688, 901, 155, 333, 893, 420, 67]
  Non-IID seeds: [27, 1165, 8965, 688, 901, 155, 333, 893, 420, 67]
  PT root      : /content/drive/MyDrive/Thesis Dataset/results/SELECTED_SEEDS_PT


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 3 — IMPORTS, CLASSES, HELPERS
# ═══════════════════════════════════════════════════════════════════════════
import copy, random, os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

class ClientDataset(Dataset):
    def __init__(self, df):
        for col in df.columns:
            if col != 'label_encoded':
                df[col] = pd.to_numeric(df[col], errors='coerce')
        df = df.fillna(0)
        self.X = torch.tensor(df.drop(['label', 'label_encoded'], axis=1).values, dtype=torch.float32)
        self.y = torch.tensor(df['label_encoded'].values, dtype=torch.float32).unsqueeze(1)
    def __len__(self): return len(self.y)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]

class MLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 1)
    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        return self.fc3(x)

def load_client_data(num_clients, distribution, base_path):
    # distribution: 'IID'     -> scen-iid-client{i}.csv
    #               'Non-IID' -> scen-noniid-client{i}.csv
    tag = 'IID' if distribution == 'IID' else 'nonIID'
    client_dfs = []
    for i in range(1, num_clients + 1):
        path = f'{base_path}/client{i}-{tag}.csv'
        if not os.path.exists(path):
            raise FileNotFoundError(f'Client CSV not found: {path}')
        client_dfs.append(pd.read_csv(path))
    return client_dfs

def train_client(client_df, global_model, loss_fn, input_dim, lr,
                 batch_size, local_epochs, clip_norm):
    dataset = ClientDataset(client_df)
    client_model = copy.deepcopy(global_model)
    optimizer = torch.optim.Adam(client_model.parameters(), lr=lr)
    client_model.train()
    indices = list(range(min(batch_size, len(dataset))))
    X_batch = torch.stack([dataset[i][0] for i in indices])
    y_batch = torch.stack([dataset[i][1] for i in indices])
    for _ in range(local_epochs):
        optimizer.zero_grad()
        pred = client_model(X_batch)
        loss = loss_fn(pred, y_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(client_model.parameters(), clip_norm)
        optimizer.step()
    update = {k: client_model.state_dict()[k] - global_model.state_dict()[k]
              for k in global_model.state_dict()}
    return client_model, update, loss.item()

def get_model_paths(distribution, seed):
    # Returns (fl_path, dpfl_path, apem_path) using your exact folder + filename structure
    suffix = 'IID' if distribution == 'IID' else 'nonIID'
    folder = os.path.join(PT_ROOT, distribution, str(seed))
    return (
        os.path.join(folder, f'Copy of fl_{suffix}.pt'),
        os.path.join(folder, f'Copy of dpfl_{suffix}.pt'),
        os.path.join(folder, f'Copy of apem_{suffix}.pt'),
    )

print('Classes and helpers defined.')

Classes and helpers defined.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 4 — CORRECTED ATTACK FUNCTIONS
# The ONLY change vs. the original notebook: dummy_y is now random 0/1.
# ═══════════════════════════════════════════════════════════════════════════

def capture_client_gradient(client_model, loss_fn, client_df, batch_size):
    """Capture the gradient from a client's intercepted update (real data)."""
    dataset = ClientDataset(client_df)
    indices = list(range(min(batch_size, len(dataset))))
    X_batch = torch.stack([dataset[i][0] for i in indices])
    y_batch = torch.stack([dataset[i][1] for i in indices])
    client_model.eval()
    client_model.zero_grad()
    loss = loss_fn(client_model(X_batch), y_batch)
    loss.backward()
    grad_flat = torch.cat([
        p.grad.clone().detach().flatten()
        for p in client_model.parameters() if p.grad is not None
    ])
    client_model.zero_grad()
    return grad_flat


def compute_reference_gradient(client_model, loss_fn, input_dim, batch_size):
    """
    Attacker's reference gradient from random dummy data.

    FIX: dummy_y is now random 0/1 (was all zeros).
    All-zero labels made the reference always point in the same direction
    (gradient of 'predict benign for random inputs'), which caused FL to
    appear most negative — opposite of the correct ordering.
    Random labels produce a neutral reference with no systematic direction.
    """
    dummy_X = torch.randn(batch_size, input_dim)
    dummy_y = torch.randint(0, 2, (batch_size, 1)).float()   # FIXED: was torch.zeros(batch_size, 1)
    client_model.eval()
    client_model.zero_grad()
    loss = loss_fn(client_model(dummy_X), dummy_y)
    loss.backward()
    ref_grad_flat = torch.cat([
        p.grad.clone().detach().flatten()
        for p in client_model.parameters() if p.grad is not None
    ])
    client_model.zero_grad()
    return ref_grad_flat


def run_corrected_attack(client_dfs, fl_model, dpfl_model, apem_model,
                         loss_fn, input_dim, apem_final_noise,
                         attack_batch_size=32, n_dummy_trials=5):
    """Run corrected gradient cosine similarity attack on all 3 models."""
    def _attack_one_model(trained_model, noise_sigmas, label):
        per_client = []
        client_results = {}
        for client_idx, client_df in enumerate(client_dfs):
            client_label = f'C{client_idx + 1}'
            client_model, _, _ = train_client(
                client_df, trained_model, loss_fn,
                input_dim, LR, attack_batch_size, LOCAL_EPOCHS, CLIP_NORM,
            )
            if noise_sigmas is not None:
                sigma = noise_sigmas[client_idx]
                noisy_model = copy.deepcopy(client_model)
                with torch.no_grad():
                    for param in noisy_model.parameters():
                        param.add_(torch.normal(mean=0.0, std=sigma * CLIP_NORM, size=param.shape))
                attack_model = noisy_model
            else:
                attack_model = copy.deepcopy(client_model)

            real_grad = capture_client_gradient(attack_model, loss_fn, client_df, attack_batch_size)
            trial_sims = []
            for _ in range(n_dummy_trials):
                ref_grad = compute_reference_gradient(attack_model, loss_fn, input_dim, attack_batch_size)
                cos_sim = F.cosine_similarity(real_grad.unsqueeze(0), ref_grad.unsqueeze(0)).item()
                trial_sims.append(cos_sim)

            mean_sim = float(np.mean(trial_sims))
            std_sim  = float(np.std(trial_sims))
            per_client.append(mean_sim)
            client_results[client_label] = {
                'mean_cos_sim': mean_sim, 'std_cos_sim': std_sim,
                'trials': trial_sims,
                'n_samples': min(attack_batch_size, len(ClientDataset(client_df))),
                'sigma_used': noise_sigmas[client_idx] if noise_sigmas else 0.0,
            }
            sigma_str = 'none' if noise_sigmas is None else f'{noise_sigmas[client_idx]:.4f}'
            print(f'    [{client_label}] cos_sim={mean_sim:+.6f} +/- {std_sim:.6f}  sigma={sigma_str}')

        overall_mean = float(np.mean(per_client))
        overall_std  = float(np.std(per_client))
        print(f'  [{label}] Overall mean cos_sim = {overall_mean:+.6f} +/- {overall_std:.6f}')
        return per_client, client_results, overall_mean

    print('\n  [1/3] Attacking FL (no noise)...')
    fl_per, fl_res, fl_mean = _attack_one_model(fl_model, None, 'FL')

    dpfl_sigma = [NOISE_MULTIPLIER] * NUM_CLIENTS
    print(f'\n  [2/3] Attacking Static DP-FL (sigma={NOISE_MULTIPLIER} all clients)...')
    dpfl_per, dpfl_res, dpfl_mean = _attack_one_model(dpfl_model, dpfl_sigma, 'DP-FL')

    print(f'\n  [3/3] Attacking APEM (per-client adaptive sigma)...')
    apem_per, apem_res, apem_mean = _attack_one_model(apem_model, apem_final_noise, 'APEM')

    results = {'FL': fl_res, 'DPFL': dpfl_res, 'APEM': apem_res}
    means   = {'FL': fl_mean, 'DPFL': dpfl_mean, 'APEM': apem_mean}
    return results, means

print('Corrected attack functions defined.')

Corrected attack functions defined.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 5 — QUICK FILE CHECK
# Run this first to confirm all .pt files are found before starting.
# ═══════════════════════════════════════════════════════════════════════════

print('Checking all expected .pt files...\n')
all_ok = True
for dist, seeds in [('IID', SEEDS_IID), ('Non-IID', SEEDS_NONIID)]:
    for seed in seeds:
        fl_p, dpfl_p, apem_p = get_model_paths(dist, seed)
        for path, name in [(fl_p, 'FL'), (dpfl_p, 'DP-FL'), (apem_p, 'APEM')]:
            exists = os.path.exists(path)
            status = 'OK' if exists else 'NOT FOUND'
            print(f'  [{status}]  [{dist} seed {seed}] {name}: {os.path.basename(path)}')
            if not exists:
                print(f'         Full path: {path}')
                all_ok = False

print()
if all_ok:
    print('All files found. Ready to run the attack in Cell 6.')
else:
    print('Some files are missing.')
    print('Check the full paths printed above against your Drive.')
    print('If the filenames differ slightly, update get_model_paths() in Cell 3.')
    print()
    print('Tip: run the line below to list what is actually in a seed folder:')
    print('  import os; print(os.listdir("<your seed folder path>"))')

Checking all expected .pt files...

  [OK]  [IID seed 27] FL: Copy of fl_IID.pt
  [OK]  [IID seed 27] DP-FL: Copy of dpfl_IID.pt
  [OK]  [IID seed 27] APEM: Copy of apem_IID.pt
  [OK]  [IID seed 1165] FL: Copy of fl_IID.pt
  [OK]  [IID seed 1165] DP-FL: Copy of dpfl_IID.pt
  [OK]  [IID seed 1165] APEM: Copy of apem_IID.pt
  [OK]  [IID seed 8965] FL: Copy of fl_IID.pt
  [OK]  [IID seed 8965] DP-FL: Copy of dpfl_IID.pt
  [OK]  [IID seed 8965] APEM: Copy of apem_IID.pt
  [OK]  [IID seed 688] FL: Copy of fl_IID.pt
  [OK]  [IID seed 688] DP-FL: Copy of dpfl_IID.pt
  [OK]  [IID seed 688] APEM: Copy of apem_IID.pt
  [OK]  [IID seed 901] FL: Copy of fl_IID.pt
  [OK]  [IID seed 901] DP-FL: Copy of dpfl_IID.pt
  [OK]  [IID seed 901] APEM: Copy of apem_IID.pt
  [OK]  [IID seed 155] FL: Copy of fl_IID.pt
  [OK]  [IID seed 155] DP-FL: Copy of dpfl_IID.pt
  [OK]  [IID seed 155] APEM: Copy of apem_IID.pt
  [OK]  [IID seed 333] FL: Copy of fl_IID.pt
  [OK]  [IID seed 333] DP-FL: Copy of dpfl_IID.pt
  

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 6 — RUN CORRECTED ATTACK FOR ALL SEEDS (IID + Non-IID)
# ═══════════════════════════════════════════════════════════════════════════

loss_fn = nn.BCEWithLogitsLoss()

# Detect input_dim from any client CSV
_temp_df = pd.read_csv(f'{BASE_PATH}/client1-nonIID.csv')
input_dim = ClientDataset(_temp_df).X.shape[1]
print(f'Input dim detected: {input_dim}')

all_corrected_rows = []

for distribution, seeds, sigma_map in [
    ('IID',     SEEDS_IID,    APEM_SIGMA_IID),
    ('Non-IID', SEEDS_NONIID, APEM_SIGMA_NONIID),
]:
    print(f'\n{chr(9619)*65}')
    print(f'  DISTRIBUTION: {distribution}')
    print(f'{chr(9619)*65}')

    for seed in seeds:
        print(f'\n{chr(9608)*65}')
        print(f'  SEED {seed}')
        print(f'{chr(9608)*65}')

        set_seed(seed)

        # Load model .pt files
        fl_path, dpfl_path, apem_path = get_model_paths(distribution, seed)
        fl_model   = MLP(input_dim); fl_model.load_state_dict(torch.load(fl_path,   map_location='cpu'))
        dpfl_model = MLP(input_dim); dpfl_model.load_state_dict(torch.load(dpfl_path, map_location='cpu'))
        apem_model = MLP(input_dim); apem_model.load_state_dict(torch.load(apem_path, map_location='cpu'))
        print(f'  Models loaded from: {os.path.dirname(fl_path)}')

        # Get APEM final noise
        if seed in sigma_map:
            apem_final_noise = sigma_map[seed]
            print(f'  APEM final sigma: {[round(s,4) for s in apem_final_noise]}')
        else:
            apem_final_noise = [NOISE_MULTIPLIER] * NUM_CLIENTS
            print(f'  WARNING: No APEM sigma found for seed {seed}. Using fallback sigma={NOISE_MULTIPLIER} for all clients.')

        # Load client data
        client_dfs = load_client_data(NUM_CLIENTS, distribution, BASE_PATH)
        print(f'  Client data loaded ({NUM_CLIENTS} clients).')

        # Run corrected attack
        attack_results, attack_means = run_corrected_attack(
            client_dfs, fl_model, dpfl_model, apem_model,
            loss_fn, input_dim, apem_final_noise,
            attack_batch_size=ATTACK_BATCH_SIZE,
            n_dummy_trials=ATTACK_N_TRIALS,
        )

        # Save per-seed corrected attack CSV next to the .pt files
        seed_folder = os.path.join(PT_ROOT, distribution, str(seed))
        corrected_rows = []
        for model_label, model_res in attack_results.items():
            for client_label, vals in model_res.items():
                corrected_rows.append({
                    'distribution': distribution,
                    'seed':         seed,
                    'model':        model_label,
                    'client':       client_label,
                    'mean_cos_sim': vals['mean_cos_sim'],
                    'std_cos_sim':  vals['std_cos_sim'],
                    'sigma_used':   vals['sigma_used'],
                    'n_samples':    vals['n_samples'],
                })
        all_corrected_rows.extend(corrected_rows)

        csv_out = os.path.join(seed_folder, f'attack_CORRECTED_{distribution}_seed{seed}.csv')
        pd.DataFrame(corrected_rows).to_csv(csv_out, index=False)
        print(f'  Corrected CSV saved: {csv_out}')

        # Per-seed summary
        print(f'\n  --- Seed {seed} | {distribution} ---')
        print(f'  FL (Non-Private)  mean cos_sim : {attack_means["FL"]:+.6f}')
        print(f'  Static DP-FL      mean cos_sim : {attack_means["DPFL"]:+.6f}')
        print(f'  APEM              mean cos_sim : {attack_means["APEM"]:+.6f}')
        fl_gt_dpfl   = attack_means['FL']   > attack_means['DPFL']
        dpfl_gt_apem = attack_means['DPFL'] > attack_means['APEM']
        print(f'  FL > DP-FL   : {"CORRECT" if fl_gt_dpfl   else "INVERTED -- check APEM sigma or data"}')
        print(f'  DP-FL > APEM : {"CORRECT" if dpfl_gt_apem else "INVERTED -- check APEM sigma or data"}')

print('\n\nAll seeds done.')

Input dim detected: 39

▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
  DISTRIBUTION: IID
▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓

█████████████████████████████████████████████████████████████████
  SEED 27
█████████████████████████████████████████████████████████████████
  Models loaded from: /content/drive/MyDrive/Thesis Dataset/results/SELECTED_SEEDS_PT/IID/27
  APEM final sigma: [1.74, 1.74, 1.74, 1.74, 1.74, 1.74, 1.74, 1.74, 1.74, 1.74]
  Client data loaded (10 clients).

  [1/3] Attacking FL (no noise)...
    [C1] cos_sim=-0.521055 +/- 0.030537  sigma=none
    [C2] cos_sim=-0.765687 +/- 0.021093  sigma=none
    [C3] cos_sim=-0.338011 +/- 0.025717  sigma=none
    [C4] cos_sim=-0.793924 +/- 0.020679  sigma=none
    [C5] cos_sim=-0.732048 +/- 0.023695  sigma=none
    [C6] cos_sim=-0.812881 +/- 0.032974  sigma=none
    [C7] cos_sim=-0.691446 +/- 0.042818  sigma=none
    [C8] cos_sim=-0.765592 +/- 0.025250  sigma=none
    [C9] cos_sim=-0.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 7 — FINAL SUMMARY TABLE + SAVE COMBINED CSV
# ═══════════════════════════════════════════════════════════════════════════

df_all = pd.DataFrame(all_corrected_rows)

summary = (
    df_all.groupby(['distribution', 'seed', 'model'])['mean_cos_sim']
    .mean()
    .reset_index()
)

print('\n' + '='*72)
print('  CORRECTED GRADIENT COSINE SIMILARITY -- FINAL SUMMARY')
print('='*72)
print(f"  {'Dist':<10} {'Seed':<10} {'Model':<20} {'Mean CosSim':>12}")
print('-'*72)
for _, row in summary.iterrows():
    print(f"  {row['distribution']:<10} {str(row['seed']):<10} {row['model']:<20} {row['mean_cos_sim']:>+12.6f}")
print('='*72)
print('  INTERPRETATION:')
print('  Higher (toward +1.0) = gradient direction is exploitable = LESS private')
print('  Lower  (toward  0.0) = signal destroyed by noise         = MORE private')
print('  Correct ordering per seed: FL (highest) > DP-FL > APEM (lowest)')
print('='*72)

# Save combined CSV to SELECTED_SEEDS_PT root
combined_out = os.path.join(PT_ROOT, 'attack_CORRECTED_combined_all_seeds.csv')
df_all.to_csv(combined_out, index=False)
print(f'\n  Combined CSV saved: {combined_out}')


  CORRECTED GRADIENT COSINE SIMILARITY -- FINAL SUMMARY
  Dist       Seed       Model                 Mean CosSim
------------------------------------------------------------------------
  IID        27         APEM                    -0.056239
  IID        27         DPFL                    -0.037887
  IID        27         FL                      -0.590107
  IID        67         APEM                    -0.139755
  IID        67         DPFL                    -0.094767
  IID        67         FL                      -0.730685
  IID        155        APEM                    -0.074925
  IID        155        DPFL                    -0.021352
  IID        155        FL                      -0.720627
  IID        333        APEM                    -0.117885
  IID        333        DPFL                    -0.120339
  IID        333        FL                      -0.781141
  IID        420        APEM                    -0.318946
  IID        420        DPFL                    -0.177592
